In [1]:
#Imports
import torch
import torch.nn as nn

In [2]:
#Test Data
t = torch.linspace(0, 100, 500)
series = torch.sin(t)

p = 10

X, Y = [], []
for i in range(len(series) - p):
    X.append(series[i:i+p])
    Y.append(series[i+p])

X = torch.stack(X)
Y = torch.stack(Y).unsqueeze(1)

X = X.unsqueeze(2)             
print("X shape:", X.shape)       
print("Y shape:", Y.shape)     

X shape: torch.Size([490, 10, 1])
Y shape: torch.Size([490, 1])


In [3]:
#Reversible Instance Normalization
class RevIN(nn.Module):
    def __init__(self, num_features, eps = 1.5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(num_features))
        self.affine_bias = nn.Parameter(torch.zeros(num_features))

    def forward(self, x, mode):
        if mode == "norm":
            self.mean = x.mean(dim = 1, keepdim = True).detach()
            self.std = x.std(dim = 1, keepdim = True, unbiased = False).detach()
            x = (x - self.mean) / (self.std + self.eps)
            x = x * self.affine_weight + self.affine_bias
            return x
        elif mode == "denorm":
            x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            x = x * (self.std + self.eps) + self.mean
            return x

In [8]:
#Loss and Optimizer
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

In [9]:
#Training Loop
num_epochs = 300
for epoch in range(num_epochs):
    prediction = model(X)

    loss = loss_function(prediction, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 30 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

Epoch 0, Loss: 0.4734768271446228
Epoch 30, Loss: 0.005840590689331293
Epoch 60, Loss: 0.0011097247479483485
Epoch 90, Loss: 0.0004342573811300099
Epoch 120, Loss: 0.00018545555940363556
Epoch 150, Loss: 6.314270285656676e-05
Epoch 180, Loss: 1.6411635442636907e-05
Epoch 210, Loss: 5.691309524991084e-06
Epoch 240, Loss: 3.843933427560842e-06
Epoch 270, Loss: 3.072430217798683e-06


In [15]:
#Evaluation
def evaluate(model, X, Y):
    model.eval
    with torch.no_grad():
        predictions = model(X)
        return loss_function(predictions, Y).item()

print(f"\nfinal MSE: {evaluate(model, X, Y):.6f}")


final MSE: 0.000002
